# 02 — Forest cover from Sentinel-2

**Time:** 10:35–12:00

We do **not** train a model. The operational binary conifer layer for this
tile already exists (summer 2024-08-31 + winter 2024-03-24, tile T11UMS).
You will:

1. Look at summer NDVI vs winter NDSI (why both seasons).
2. Turn binary conifer into **canopy cover** (3×3 neighbourhood, percent).
3. Turn canopy into **gap area** (opening size in hectares).

PRA uses canopy cover. The ATES classifier uses gap area. Treeline from this
layer (~1980 m here) only segments large PRAs — it is not an ATES input.


In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path(".").resolve()))
from workshop import DATA, OUT, AUTOATES_ROOT, need, read_raster, write_raster, show, canopy_from_binary
import matplotlib.pyplot as plt
import numpy as np

binary_path = need(DATA / "02_forest" / "inputs" / "sen2_forest_binary.tif")
ndvi_path = need(DATA / "02_forest" / "inputs" / "ndvi_summer.tif")
ndsi_path = need(DATA / "02_forest" / "inputs" / "ndsi_winter.tif")
binary, profile, transform, crs = read_raster(binary_path)
ndvi, *_ = read_raster(ndvi_path)
ndsi, *_ = read_raster(ndsi_path)
# indices were stored x10000
ndvi = ndvi / 10000.0
ndsi = ndsi / 10000.0
print(crs, binary.shape, "conifer fraction", np.nanmean(binary).round(3))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4.5))
show(ndvi, "Summer NDVI", "RdYlGn", vmin=0, vmax=0.8, ax=axs[0])
show(ndsi, "Winter NDSI", "Blues", vmin=-0.2, vmax=0.8, ax=axs[1])
show(binary, "Binary conifer (operational)", "Greens", vmin=0, vmax=1, ax=axs[2])
plt.tight_layout()

In [ ]:
cover = canopy_from_binary(binary, nodata_mask=~np.isfinite(binary))
cell = abs(transform.a)

sys.path.insert(0, str(AUTOATES_ROOT))
from autoates.comAutoATES.atesValidation.forestGaps import gap_metrics

valid = np.isfinite(cover)
width, area = gap_metrics(np.nan_to_num(cover, nan=100.0), cell, valid=valid,
                          max_area_ha=100.0)

work = OUT / "workshop"
write_raster(work / "canopy_cover.tif", cover, profile)
write_raster(work / "gap_area_ha.tif", area, profile)
print("gap area ha: min", float(np.nanmin(area)), "max", float(np.nanmax(area)))

fig, axs = plt.subplots(1, 2, figsize=(11, 5))
show(cover, "Canopy cover % (3×3)", "YlGn", vmin=0, vmax=100, ax=axs[0])
show(np.where(area > 0, area, np.nan), "Gap area (ha)", "YlOrRd", vmin=0, vmax=20, ax=axs[1])
plt.tight_layout()

The PRA/ATES live path uses the **production** canopy product
(`data/02_forest/outputs_reference/sen2_canopy_cover.tif`), which is the same
method run domain-wide. Your 3×3 layer is the local version of that step.

**If stuck:** copy `data/02_forest/outputs_reference/` and continue after lunch.
